# Binary grain vs background (SwinV2 + UPerNet, 5-fold CV)

This notebook is separate from `colab_run_pipeline_221.ipynb`. It trains **binary** segmentation:

- **Class 0 (non-grain / background):** original multiclass label `0`.
- **Class 1 (grain):** any original label in `1`–`15` (bivalves, micrite, cement, …, brachiopod).
- **Ignore:** original value `255` stays ignored in the loss.

Multiclass masks are converted **on the fly** in the dataset (no duplicate mask files written).

Training uses the same **SwinV2-Tiny + UPerNet** setup as the multiclass pipeline (`num_labels=2`), **5-fold** cross-validation over image/mask pairs, and logs **pixel accuracy**, **mean IoU**, and **per-class IoU** each validation epoch. Summary metrics are saved to `cv_summary.json` under `--output_dir`.

**Typical order:** install → mount Drive → **git pull** → set paths → smoke / training cells.

In [ ]:
# Install dependencies (Colab)
!pip -q install --upgrade transformers tqdm

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2b) Pull from GitHub (full git log: stash, fetch, fast-forward, file stats — same style as the SSL/Seg Colab)
import os
from pathlib import Path

REPO = Path("/content/drive/My Drive/Payne_lab_swin_transformer")
if not REPO.is_dir():
    raise FileNotFoundError(f"Clone not found: {REPO} — mount Drive and fix the path.")
os.chdir(REPO)
print(REPO.resolve(), "\n")
!git status -sb
!git stash push -m "colab-local-before-pull"
!git pull origin feat/swin-upernet-training-pipeline

In [ ]:
import os
from pathlib import Path

# Repo on Drive (update if your clone lives elsewhere)
REPO_ROOT = Path("/content/drive/My Drive/Payne_lab_swin_transformer")

# Labeled image/mask dirs (same as colab_run_pipeline_221: adjust to your dataset folder)
IMG_DIR = "/content/drive/My Drive/Petrographic images_ML work/labelled images_PS/labelledDataset_02032026/my_dataset/img"
MASK_DIR = "/content/drive/My Drive/Petrographic images_ML work/labelled images_PS/labelledDataset_02032026/my_dataset/masks_machine"

# Output roots (binary CV vs multiclass/SSL in the other notebook)
OUT_ROOT_221 = "/content/drive/My Drive/Petrographic images_ML work/model_outputs_221"
OUT_BINARY = "/content/drive/My Drive/Petrographic images_ML work/model_outputs_binary_cv"

# Export for `!python ... $IMG_DIR` cells (bash reads these; same pattern as the other Colab)
os.environ["REPO_ROOT"] = str(REPO_ROOT)
os.environ["IMG_DIR"] = IMG_DIR
os.environ["MASK_DIR"] = MASK_DIR
os.environ["OUT_BINARY"] = OUT_BINARY
os.environ["OUT_ROOT_221"] = OUT_ROOT_221

print("Repo exists:", REPO_ROOT.exists())
print("IMG dir exists:", Path(IMG_DIR).exists())
print("MASK dir exists:", Path(MASK_DIR).exists())

## Smoke test

Builds dataloaders and checks paired samples (no training).

In [ ]:
# Smoke test: dataloader only (unbuffered; logs like colab_run_pipeline_221)
import os
os.chdir(REPO_ROOT)
!python -u code/model_training_pipeline/swin_binary_segmentation_221.py \
  --img_dir "$IMG_DIR" \
  --mask_dir "$MASK_DIR" \
  --output_dir "$OUT_BINARY/smoke" \
  --no_train

## Full 5-fold cross-validation

Run the **config** cell first so `REPO_ROOT` and `os.environ` (`IMG_DIR`, `MASK_DIR`, `OUT_BINARY`, `OUT_ROOT_221`) are set.

The next cell uses **`os.chdir` + `!python -u`** (same style as `colab_run_pipeline_221` segmentation) so training logs stream like the multiclass finetune. **`--num_workers 0` is set for Colab + Drive:** worker processes often make the first minutes slower; tqdm stays at `0%` until the first batch finishes (Drive I/O + GPU), which is normal.

Each epoch now also prints the validation pixel split for **ground truth** and **prediction** (`background` vs `grain`) so you can directly see class balance and prediction collapse.

Edit or remove `--backbone_checkpoint` to point at your SSL weights (default: `$OUT_ROOT_221/ssl_full/ssl_swinv2_best.pth`).

In [ ]:
# Full binary CV (same invocation style as colab_run_pipeline_221 cell 6)
import os
os.chdir(REPO_ROOT)
!python -u code/model_training_pipeline/swin_binary_segmentation_221.py \
  --img_dir "$IMG_DIR" \
  --mask_dir "$MASK_DIR" \
  --output_dir "$OUT_BINARY/cv5" \
  --n_folds 5 \
  --epochs 20 \
  --batch_size 2 \
  --crop 512 \
  --num_workers 0 \
  --backbone_checkpoint "$OUT_ROOT_221/ssl_full/ssl_swinv2_best.pth"

In [ ]:
import csv
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

cv_dir = Path(OUT_BINARY) / "cv5"
summary_path = cv_dir / "cv_summary.json"
if not summary_path.is_file():
    raise FileNotFoundError(f"Missing summary: {summary_path}")

with summary_path.open("r", encoding="utf-8") as f:
    summary = json.load(f)

fold_rows = summary.get("per_fold_best_at_checkpoint", [])
if not fold_rows:
    raise RuntimeError("No fold rows found in cv_summary.json")

# --- Plot 1: best fold mIoU overview ---
fold_ids = [int(r["fold"]) for r in fold_rows]
best_mious = [float(r["miou"]) for r in fold_rows]
best_epochs = [int(r["epoch"]) for r in fold_rows]

order = np.argsort(fold_ids)
fold_ids = [fold_ids[i] for i in order]
best_mious = [best_mious[i] for i in order]
best_epochs = [best_epochs[i] for i in order]

plt.figure(figsize=(10, 4))
plt.bar([f"fold_{k}" for k in fold_ids], best_mious, color="#4C78A8")
for i, (m, ep) in enumerate(zip(best_mious, best_epochs)):
    plt.text(i, m + 0.01, f"ep{ep}\n{m:.3f}", ha="center", va="bottom", fontsize=9)
plt.ylim(0, min(1.0, max(best_mious) + 0.12))
plt.ylabel("Best mIoU")
plt.title("Best validation mIoU per fold")
plt.grid(axis="y", alpha=0.25)
plt.show()

print("mean_best_miou:", round(float(summary.get("mean_best_miou", 0.0)), 4))
print("std_best_miou :", round(float(summary.get("std_best_miou", 0.0)), 4))

# --- Load all fold epoch metrics ---
fold_metrics = {}
for k in fold_ids:
    p = cv_dir / f"fold_{k}" / "val_metrics.csv"
    if not p.is_file():
        print(f"[warn] Missing {p}")
        continue
    with p.open("r", encoding="utf-8") as f:
        rdr = csv.DictReader(f)
        rows = [row for row in rdr]
    fold_metrics[k] = rows

if not fold_metrics:
    raise RuntimeError("No fold val_metrics.csv files found.")

# --- Plot 2: mIoU trajectories ---
plt.figure(figsize=(10, 5))
for k, rows in sorted(fold_metrics.items()):
    x = [int(r["epoch"]) for r in rows]
    y = [float(r["miou"]) for r in rows]
    plt.plot(x, y, marker="o", ms=3, label=f"fold_{k}")
plt.xlabel("Epoch")
plt.ylabel("Validation mIoU")
plt.title("mIoU trajectory by fold")
plt.grid(alpha=0.25)
plt.legend(ncol=3, fontsize=9)
plt.show()

# --- Plot 3: background vs grain IoU by fold ---
fig, axes = plt.subplots(len(fold_metrics), 1, figsize=(11, 3.2 * len(fold_metrics)), sharex=True)
if len(fold_metrics) == 1:
    axes = [axes]

for ax, (k, rows) in zip(axes, sorted(fold_metrics.items())):
    x = [int(r["epoch"]) for r in rows]
    iou_bg = [float(r["iou_bg"]) for r in rows]
    iou_grain = [float(r["iou_grain"]) for r in rows]
    ax.plot(x, iou_bg, "-o", ms=3, label="IoU background", color="#E45756")
    ax.plot(x, iou_grain, "-o", ms=3, label="IoU grain", color="#54A24B")
    ax.set_ylabel(f"fold_{k}")
    ax.set_ylim(0, 1)
    ax.grid(alpha=0.25)
    ax.legend(loc="lower right", fontsize=8)

axes[-1].set_xlabel("Epoch")
fig.suptitle("Per-class IoU trajectories (background vs grain)", y=1.02)
plt.tight_layout()
plt.show()

# --- Plot 4: class split drift (requires new CSV columns) ---
have_split_cols = all(
    rows and {"gt_bg_pct", "gt_grain_pct", "pred_bg_pct", "pred_grain_pct"}.issubset(rows[0].keys())
    for rows in fold_metrics.values()
)

if have_split_cols:
    fig, axes = plt.subplots(len(fold_metrics), 1, figsize=(11, 3.2 * len(fold_metrics)), sharex=True)
    if len(fold_metrics) == 1:
        axes = [axes]

    for ax, (k, rows) in zip(axes, sorted(fold_metrics.items())):
        x = [int(r["epoch"]) for r in rows]
        gt_bg = [float(r["gt_bg_pct"]) for r in rows]
        pred_bg = [float(r["pred_bg_pct"]) for r in rows]
        ax.plot(x, gt_bg, "-", lw=2, label="GT background %", color="#4C78A8")
        ax.plot(x, pred_bg, "--", lw=2, label="Pred background %", color="#F58518")
        ax.set_ylabel(f"fold_{k}\n% bg")
        ax.set_ylim(0, 100)
        ax.grid(alpha=0.25)
        ax.legend(loc="best", fontsize=8)

    axes[-1].set_xlabel("Epoch")
    fig.suptitle("Background split drift (GT vs Prediction)", y=1.02)
    plt.tight_layout()
    plt.show()
else:
    print("Split % columns not found in val_metrics.csv (generated by older script version).")
    print("Re-run training with the latest script to plot GT vs prediction split over epochs.")